<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/01_stageI_eqv2_adsorbml_relaxation_article_repository.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/01_stageI_eqv2_adsorbml_relaxation_article_repository.ipynb"
target="_parent"> <img src="https://colab.research.google.com/assets/colab-badge.svg"
    alt="Open in Colab"/> </a>

# Stage I — AdsorbML + EquiformerV2 Relaxation and VASP Export

This repository-ready Google Colab notebook demonstrates the first stage of the oxygen reduction reaction (ORR) screening workflow for the representative material **mp-10260(111)**.

The workflow combines AdsorbML structure generation with EquiformerV2 relaxation, trajectory validation, adsorption-site analysis, minimum-energy structure selection, and repository-ready VASP input generation.

## Notebook Workflow

1. Install the required legacy FAIRChem environment and restart the runtime.
2. Verify the software versions and import the required packages.
3. Download the official Open Catalyst bulk and adsorbate databases.
4. Select `mp-10260` and generate its `(111)` slab terminations.
5. Construct the `O*`, `OH*`, and `OOH*` adsorbates.
6. Generate heuristic adsorption sites together with 20 randomly sampled configurations per adsorbate.
7. Load the `EquiformerV2-31M-S2EF-OC20-All+MD` checkpoint.
8. Relax the constrained bare slab and all adsorbate–slab configurations independently.
9. Validate the resulting relaxation trajectories.
10. Classify the final adsorption sites and remove converged duplicate structures.
11. Visualize the relaxed structures using py3Dmol.
12. Export the summary tables, final structures, and repository-ready VASP input folders.

## Workflow Summary

The notebook performs the following main tasks:

1. Generates and independently relaxes the constrained clean slab using
   `EquiformerV2-31M-S2EF-OC20-All+MD`.

2. Loads `O*` and `OH*` from the Open Catalyst adsorbate database and constructs
   `OOH*` explicitly.

3. Enumerates AdsorbML heuristic atop, bridge, and hollow adsorption sites.

4. Adds 20 randomly sampled adsorption configurations for each adsorbate.

5. Relaxes every adsorbate–slab configuration using EqV2 and the BFGS optimizer until

$$
F_{\max} < 0.05\ \mathrm{eV} \mathring{\mathrm{A}}^{-1},
$$

or until the predefined maximum number of optimization steps is reached.

6. Validates the trajectories, classifies the final adsorption sites, removes
   converged duplicate structures, and identifies the lowest-scoring configurations
   for `O*`, `OH*`, and `OOH*`.

7. Exports the final EqV2 frame of the relaxed bare slab and each adsorbate global
   minimum into separate `vasp/` directories containing:

   * `POSCAR`
   * `INCAR`
   * `KPOINTS`
   * `job.sh`
   * `metadata.json`
   * `POTCAR.spec`
   * a licensing-safe `POTCAR` placeholder

> **Energy interpretation**
>
> EqV2 values are used only to rank the relaxed structures. They are
> machine-learning model scores and must not be interpreted as DFT adsorption
> energies. The independently evaluated bare-slab score is recorded for
> provenance but is not subtracted from the adsorbate–slab scores in this
> notebook.

> **POTCAR licensing**
>
> VASP PAW datasets are not redistributed in this repository. Each exported
> calculation directory contains a deliberately non-runnable `POTCAR`
> placeholder and a `POTCAR.spec` file defining the required elemental order.
> The placeholder must be replaced privately with a properly licensed `POTCAR`
> before running VASP.

## Method Represented in This Notebook

For each slab–adsorbate system,

$$
X^{} \in {\mathrm{O}^{},\ \mathrm{OH}^{},\ \mathrm{OOH}^{}},
$$

the initial adsorption geometries are generated using AdsorbML through a two-step protocol.
protocol.

First, heuristic atop, bridge, and hollow adsorption candidates are enumerated
from the surface geometry. Next, an additional

$$
N_{\mathrm{random}} = 20
$$

configurations are sampled from the surface Delaunay triangulation using
`random_site_heuristic_placement`.

This random-site procedure preserves the designated adsorbate binding atom and
applies a randomized azimuthal orientation together with a small molecular tilt.
It therefore avoids the center-of-mass placement used in the fully random
AdsorbML mode.

The constrained clean slab is relaxed independently using the same EqV2
calculator. All adsorbate configurations are subsequently relaxed using
`EquiformerV2-31M-S2EF-OC20-All+MD` and the BFGS optimizer until

$$
F_{\max} < 0.05\ \mathrm{eV} \mathring{\mathrm{A}}^{-1},
$$

or until the predefined maximum number of optimization steps is reached.

After trajectory validation and duplicate removal, the final VASP package
contains:

* the EqV2-relaxed bare slab;
* the global-minimum `O*` configuration;
* the global-minimum `OH*` configuration; and
* the global-minimum `OOH*` configuration.

In every case, the exported `POSCAR` is generated from the final frame of the
corresponding EqV2 relaxation trajectory.


## 1. Colab environment

Choose **Runtime → Change runtime type → T4 GPU** before running the notebook.

The OC20 EqV2 checkpoint belongs to the legacy FAIRChem v1 stack. This notebook
therefore pins PyTorch, NumPy, ASE, FAIRChem, and `fairchem-data-oc` to a
compatible set. Colab may print dependency warnings for unrelated preinstalled
packages that require NumPy 2. Those packages are not used here.

The installation cell writes a marker under `/content`. After the packages are
installed, run the following restart cell exactly once.


In [1]:
# Install once per fresh Colab runtime.
# Re-running this cell after the kernel restart is safe: the marker prevents
# another multi-gigabyte installation.

from pathlib import Path
import subprocess
import sys

ENV_MARKER = Path("/content/.eqv2_legacy_environment_v2_installed")
RESTART_MARKER = Path("/content/.eqv2_legacy_kernel_v2_restarted")

if ENV_MARKER.exists():
    print("Compatible EqV2 environment is already installed in this runtime.")
else:
    commands = [
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--upgrade", "pip", "setuptools", "wheel",
        ],
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--force-reinstall",
            "torch==2.3.0",
            "--index-url", "https://download.pytorch.org/whl/cu121",
        ],
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--force-reinstall",
            "numpy==1.26.4",
            "scipy==1.13.1",
            "pandas==2.2.2",
            "ase==3.23.0",
            "pymatgen==2024.10.29",
        ],
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            "torch-geometric==2.5.3",
        ],
        [
            # Compiled PyTorch Geometric extensions required by legacy FAIRChem.
            # These wheels must match PyTorch 2.3.0 and CUDA 12.1 exactly.
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            "pyg_lib",
            "torch_scatter==2.1.2+pt23cu121",
            "torch_sparse==0.6.18+pt23cu121",
            "torch_cluster==1.6.3+pt23cu121",
            "torch_spline_conv==1.2.2+pt23cu121",
            "-f", "https://data.pyg.org/whl/torch-2.3.0+cu121.html",
        ],
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            "fairchem-core==1.2.0",
            "fairchem-data-oc==0.2.0",
            "py3Dmol>=2.0,<3",
            "ipywidgets>=8,<9",
        ],
    ]

    for command in commands:
        print("\nRunning:", " ".join(command))
        subprocess.check_call(command)

    ENV_MARKER.write_text("installed\n", encoding="utf-8")
    RESTART_MARKER.unlink(missing_ok=True)

    print("\nInstallation completed.")
    print("Now run the next cell to restart the Python kernel once.")



Running: /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel

Running: /usr/bin/python3 -m pip install --no-cache-dir --force-reinstall torch==2.3.0 --index-url https://download.pytorch.org/whl/cu121

Running: /usr/bin/python3 -m pip install --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.13.1 pandas==2.2.2 ase==3.23.0 pymatgen==2024.10.29

Running: /usr/bin/python3 -m pip install --no-cache-dir torch-geometric==2.5.3

Running: /usr/bin/python3 -m pip install --no-cache-dir pyg_lib torch_scatter==2.1.2+pt23cu121 torch_sparse==0.6.18+pt23cu121 torch_cluster==1.6.3+pt23cu121 torch_spline_conv==1.2.2+pt23cu121 -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

Running: /usr/bin/python3 -m pip install --no-cache-dir fairchem-core==1.2.0 fairchem-data-oc==0.2.0 py3Dmol>=2.0,<3 ipywidgets>=8,<9

Installation completed.
Now run the next cell to restart the Python kernel once.


### Why the kernel must be killed once

`pip` changes the package files on disk, but the current Python process can
still hold the old NumPy, SciPy, or PyTorch modules in memory. That mixed state
caused the earlier `numpy.rec` and version-assertion errors.

`os.kill(os.getpid(), 9)` terminates only the current notebook kernel. Colab
automatically reconnects to a fresh Python process that imports the newly
installed versions. Files under `/content` and the notebook itself are not
deleted. A marker prevents a restart loop when the notebook is run again.


In [ ]:
# One-time kernel restart after installation.
# The marker is written before termination, so "Run all" does not create a loop.

from pathlib import Path
import os
import time

ENV_MARKER = Path("/content/.eqv2_legacy_environment_v2_installed")
RESTART_MARKER = Path("/content/.eqv2_legacy_kernel_v2_restarted")

if not ENV_MARKER.exists():
    raise RuntimeError("Run the installation cell first.")

if RESTART_MARKER.exists():
    print("Kernel restart has already been completed. Continue below.")
else:
    RESTART_MARKER.write_text("restarted\n", encoding="utf-8")
    print("Restarting the Python kernel now. Colab will reconnect automatically.")
    time.sleep(1)
    os.kill(os.getpid(), 9)


Restarting the Python kernel now. Colab will reconnect automatically.


After Colab reconnects, continue from the next cell. Running the notebook from
the top is also safe because the installation and restart markers will skip the
completed steps.


In [ ]:
# Verify the active environment after the restart.

import importlib.metadata as metadata
import sys

import ase
import numpy as np
import scipy
import torch
import torch_geometric
import pyg_lib
import torch_cluster
import torch_scatter
import torch_sparse
import torch_spline_conv

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("ASE:", ase.__version__)
print("torch-geometric:", torch_geometric.__version__)
print("pyg_lib: imported")
print("torch_scatter:", metadata.version("torch-scatter"))
print("torch_sparse:", metadata.version("torch-sparse"))
print("torch_cluster:", metadata.version("torch-cluster"))
print("torch_spline_conv:", metadata.version("torch-spline-conv"))
print("fairchem-core:", metadata.version("fairchem-core"))
print("fairchem-data-oc:", metadata.version("fairchem-data-oc"))

assert np.__version__ == "1.26.4"
assert scipy.__version__ == "1.13.1"
assert torch.__version__.split("+")[0] == "2.3.0"
assert metadata.version("torch-scatter").startswith("2.1.2")
assert metadata.version("torch-sparse").startswith("0.6.18")
assert metadata.version("torch-cluster").startswith("1.6.3")
assert metadata.version("torch-spline-conv").startswith("1.2.2")
assert metadata.version("fairchem-core") == "1.2.0"
assert metadata.version("fairchem-data-oc") == "0.2.0"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select Runtime → Change runtime type → T4 GPU."
    )

print("\nEnvironment verification passed.")


In [1]:
# Imports and global settings.

import hashlib
import io
import itertools
import json
import math
import random
import re
import shutil
import time
import warnings

from pathlib import Path

import ase.io as aseio
import ipywidgets as widgets
import numpy as np
import pandas as pd
import py3Dmol
import requests
import torch

from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
from ase.constraints import FixAtoms
from ase.data import atomic_numbers, covalent_radii
from ase.optimize import BFGS
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file
from fairchem.data.oc.core import Adsorbate, AdsorbateSlabConfig, Bulk, Slab
from fairchem.data.oc.utils import DetectTrajAnomaly

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BULK_ID = "mp-10260"
MILLER_INDEX = (1, 1, 1)
SURFACE_INDEX = 0

RANDOM_SITES_PER_ADSORBATE = 20
FMAX_THRESHOLD = 0.05
MAX_RELAX_STEPS = 200
OVERWRITE_EXISTING = False

MODEL_NAME = "EquiformerV2-31M-S2EF-OC20-All+MD"
ADSORBATE_NAMES = ("O", "OH", "OOH")
ADSORBATE_SIZES = {"O": 1, "OH": 2, "OOH": 3}

DATA_DIR = Path("/content/oc_databases")
CHECKPOINT_DIR = Path("/content/fairchem_checkpoints")
ROOT_DIR = Path("/content/stageI_eqv2_results")
SYSTEM_ID = f"{BULK_ID}_{''.join(map(str, MILLER_INDEX))}_term{SURFACE_INDEX}"
SYSTEM_DIR = ROOT_DIR / SYSTEM_ID
TRAJECTORY_DIR = SYSTEM_DIR / "trajectories"
LOG_DIR = SYSTEM_DIR / "logs"
POSCAR_DIR = SYSTEM_DIR / "poscars"
TABLE_DIR = SYSTEM_DIR / "tables"
VASP_DIR = SYSTEM_DIR / "vasp"

KPOINT_DENSITY = 40.0
VASP_RUN_MODE = "single_point"

for directory in (
    DATA_DIR,
    CHECKPOINT_DIR,
    ROOT_DIR,
    SYSTEM_DIR,
    TRAJECTORY_DIR,
    LOG_DIR,
    POSCAR_DIR,
    TABLE_DIR,
    VASP_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("System:", SYSTEM_ID)
print("Random-site placements per adsorbate:", RANDOM_SITES_PER_ADSORBATE)
print("Maximum BFGS steps:", MAX_RELAX_STEPS)
print("Force threshold:", FMAX_THRESHOLD, "eV/Å")


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


System: mp-10260_111_term0
Random-site placements per adsorbate: 20
Maximum BFGS steps: 200
Force threshold: 0.05 eV/Å


## 2. Download the OC databases

`InDomainBulk.pkl` is not required. The notebook first tries the `bulks.pkl`
file in the archived Open Catalyst Dataset repository supplied for this
workflow. If GitHub returns an HTML page, an LFS pointer, or an incomplete
download, it falls back to the official flat OC bulk database.

O* and OH* are loaded from the official OC adsorbate database. OOH* is
constructed explicitly later because it is not used here as a database lookup.

Only load pickle files from sources you trust. The URLs below are official Open
Catalyst Project resources.


In [2]:
BULK_DB_PATH = DATA_DIR / "bulks.pkl"
ADSORBATE_DB_PATH = DATA_DIR / "adsorbates.pkl"

BULK_URLS = [
    # User-supplied GitHub source.
    "https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/"
    "raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl",
    "https://raw.githubusercontent.com/Open-Catalyst-Project/"
    "Open-Catalyst-Dataset/main/ocdata/databases/pkls/bulks.pkl",
    # Official OC20 flat database fallback.
    "https://dl.fbaipublicfiles.com/opencatalystproject/data/"
    "input_generation/bulk_db_flat_2021sep20.pkl",
]

ADSORBATE_URLS = [
    "https://dl.fbaipublicfiles.com/opencatalystproject/data/"
    "input_generation/adsorbate_db_2021apr28.pkl",
]

ADSORBATE_MD5 = "975e00a62c7b634b245102e42167b3fb"
FLAT_BULK_MD5 = "43df744a8366c25392e3072d445016b2"


def md5sum(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def invalid_download(path: Path) -> bool:
    if not path.exists() or path.stat().st_size < 1024:
        return True

    with path.open("rb") as handle:
        head = handle.read(512).lower()

    return (
        b"version https://git-lfs.github.com/spec" in head
        or b"<!doctype html" in head
        or b"<html" in head
    )


def download_with_fallback(
    urls: list[str],
    target: Path,
    expected_md5_by_url: dict[str, str] | None = None,
) -> str:
    expected_md5_by_url = expected_md5_by_url or {}

    if target.exists() and not invalid_download(target):
        print(f"Using existing file: {target} ({target.stat().st_size / 1e6:.1f} MB)")
        return "existing"

    target.unlink(missing_ok=True)
    temporary = target.with_suffix(target.suffix + ".part")

    errors = []

    for url in urls:
        temporary.unlink(missing_ok=True)
        print("\nDownloading:", url)

        try:
            with requests.get(url, stream=True, timeout=(30, 300)) as response:
                response.raise_for_status()
                total = int(response.headers.get("content-length", 0))

                with temporary.open("wb") as handle, tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    desc=target.name,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))

            if invalid_download(temporary):
                raise ValueError("The downloaded object is HTML, an LFS pointer, or too small.")

            expected_md5 = expected_md5_by_url.get(url)
            if expected_md5:
                observed_md5 = md5sum(temporary)
                if observed_md5 != expected_md5:
                    raise ValueError(
                        f"MD5 mismatch: expected {expected_md5}, got {observed_md5}"
                    )

            temporary.replace(target)
            print(f"Saved: {target} ({target.stat().st_size / 1e6:.1f} MB)")
            return url

        except Exception as exc:
            errors.append(f"{url}: {type(exc).__name__}: {exc}")
            print("Failed:", errors[-1])

    raise RuntimeError(
        "Every database download failed:\n" + "\n".join(errors)
    )


bulk_source = download_with_fallback(
    BULK_URLS,
    BULK_DB_PATH,
    expected_md5_by_url={
        BULK_URLS[-1]: FLAT_BULK_MD5,
    },
)

adsorbate_source = download_with_fallback(
    ADSORBATE_URLS,
    ADSORBATE_DB_PATH,
    expected_md5_by_url={
        ADSORBATE_URLS[0]: ADSORBATE_MD5,
    },
)

print("\nBulk source:", bulk_source)
print("Adsorbate source:", adsorbate_source)



Downloading: https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl


bulks.pkl:   0%|          | 0.00/36.1M [00:00<?, ?B/s]

Saved: /content/oc_databases/bulks.pkl (36.1 MB)

Downloading: https://dl.fbaipublicfiles.com/opencatalystproject/data/input_generation/adsorbate_db_2021apr28.pkl


adsorbates.pkl:   0%|          | 0.00/57.0k [00:00<?, ?B/s]

Saved: /content/oc_databases/adsorbates.pkl (0.1 MB)

Bulk source: https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl
Adsorbate source: https://dl.fbaipublicfiles.com/opencatalystproject/data/input_generation/adsorbate_db_2021apr28.pkl


## 3. Generate the initial `mp-10260(111)` slab

The bulk is selected by Materials Project identifier, not by database position.
The code explicitly checks the returned ID because the legacy `Bulk` class can
otherwise fall back to a random material when an ID is absent.

The generated slab is saved before relaxation for provenance. After the EqV2
checkpoint is loaded, the same constrained slab is relaxed independently and
its final trajectory frame is exported as the repository `vasp/bare/POSCAR`.
The adsorbate placement step remains based on the reproducibly generated initial
slab, matching the AdsorbML screening protocol represented here.


Support for third party widgets will remain active for the duration of the session. To disable support:

In [3]:
from google.colab import output
output.disable_custom_widget_manager()

In [4]:
def rebuild_atoms(
    atoms: Atoms,
    *,
    constrain_subsurface: bool = False,
    molecule: bool = False,
) -> Atoms:
    # Recreate an ASE object to avoid incompatibilities from old pickled ASE objects.
    tags = np.asarray(atoms.get_tags(), dtype=int)

    rebuilt = Atoms(
        numbers=np.asarray(atoms.get_atomic_numbers(), dtype=int),
        positions=np.asarray(atoms.get_positions(), dtype=float),
        cell=np.asarray(atoms.cell.array, dtype=float),
        pbc=False if molecule else np.asarray(atoms.get_pbc(), dtype=bool),
        tags=tags,
    )

    if constrain_subsurface:
        fixed_indices = np.flatnonzero(tags == 0)
        if len(fixed_indices):
            rebuilt.set_constraint(FixAtoms(indices=fixed_indices))

    return rebuilt


def ensure_finite_geometry(atoms: Atoms, label: str) -> None:
    if not np.isfinite(atoms.positions).all():
        raise ValueError(f"Non-finite positions in {label}.")
    if not np.isfinite(atoms.cell.array).all():
        raise ValueError(f"Non-finite cell in {label}.")


bulk = Bulk(
    bulk_src_id_from_db=BULK_ID,
    bulk_db_path=str(BULK_DB_PATH),
)

if bulk.src_id != BULK_ID:
    raise RuntimeError(
        f"Requested {BULK_ID}, but the database returned {bulk.src_id}. "
        "The requested material is missing from this database."
    )

bulk.atoms = rebuild_atoms(bulk.atoms)

generated_slabs = Slab.from_bulk_get_specific_millers(
    bulk=bulk,
    specific_millers=MILLER_INDEX,
)

slab_candidates = (
    [generated_slabs]
    if isinstance(generated_slabs, Slab)
    else list(generated_slabs)
)

if not slab_candidates:
    raise RuntimeError(f"No {MILLER_INDEX} slab was generated for {BULK_ID}.")

if not 0 <= SURFACE_INDEX < len(slab_candidates):
    raise IndexError(
        f"SURFACE_INDEX={SURFACE_INDEX} but only "
        f"{len(slab_candidates)} terminations were generated."
    )

raw_slab_object = slab_candidates[SURFACE_INDEX]
clean_slab = rebuild_atoms(
    raw_slab_object.atoms,
    constrain_subsurface=True,
)
ensure_finite_geometry(clean_slab, "clean slab")

# Rebuild the Slab wrapper around the current ASE object.
slab_object = Slab(
    bulk=raw_slab_object.bulk,
    slab_atoms=clean_slab,
    millers=raw_slab_object.millers,
    shift=raw_slab_object.shift,
    top=raw_slab_object.top,
    oriented_bulk=raw_slab_object.oriented_bulk,
)

clean_traj_path = SYSTEM_DIR / "clean_slab_unrelaxed.traj"
clean_poscar_path = POSCAR_DIR / "clean_slab_unrelaxed" / "POSCAR"
clean_poscar_path.parent.mkdir(parents=True, exist_ok=True)

aseio.write(clean_traj_path, clean_slab)
aseio.write(
    clean_poscar_path,
    clean_slab,
    format="vasp",
    direct=True,
    vasp5=True,
    sort=True,
)

tag_counts = (
    pd.Series(clean_slab.get_tags(), name="count")
    .value_counts()
    .sort_index()
    .rename_axis("OC_tag")
    .reset_index(name="count")
)

print("Bulk ID:", bulk.src_id)
print("Bulk formula:", bulk.atoms.get_chemical_formula())
print("Requested Miller index:", MILLER_INDEX)
print("Generated terminations:", len(slab_candidates))
print("Selected termination:", SURFACE_INDEX)
print("Slab formula:", clean_slab.get_chemical_formula())
print("Number of slab atoms:", len(clean_slab))
print("Saved clean slab:", clean_poscar_path)
display(tag_counts)


Bulk ID: mp-10260
Bulk formula: Ni3Sb
Requested Miller index: (1, 1, 1)
Generated terminations: 4
Selected termination: 0
Slab formula: Ni36Sb12
Number of slab atoms: 48
Saved clean slab: /content/stageI_eqv2_results/mp-10260_111_term0/poscars/clean_slab_unrelaxed/POSCAR


,OC_tag,count
0,0,36
1,1,12


## 4. Define O*, OH*, and OOH*

- O* is loaded from the OC adsorbate database using `*O`.
- OH* is loaded from the OC adsorbate database using `*OH`.
- OOH* is constructed with the binding O at index 0.

The OOH coordinates use an O–O distance of 1.45 Å and an approximately
0.99 Å terminal O–H bond. AdsorbML binds the first oxygen to the sampled site.


In [5]:
def database_adsorbate(smiles: str) -> Adsorbate:
    adsorbate = Adsorbate(
        adsorbate_smiles_from_db=smiles,
        adsorbate_db_path=str(ADSORBATE_DB_PATH),
    )
    if adsorbate.smiles != smiles:
        raise RuntimeError(
            f"Requested adsorbate {smiles}, but database returned {adsorbate.smiles}."
        )
    adsorbate.atoms = rebuild_atoms(adsorbate.atoms, molecule=True)
    return adsorbate


adsorbate_O = database_adsorbate("*O")
adsorbate_OH = database_adsorbate("*OH")

ooh_atoms = Atoms(
    symbols=["O", "O", "H"],
    positions=[
        [0.000, 0.000, 0.000],  # binding O
        [0.000, 0.000, 1.450],  # distal O
        [0.940, 0.000, 1.750],  # terminal H
    ],
    pbc=False,
)

adsorbate_OOH = Adsorbate(
    adsorbate_atoms=ooh_atoms,
    adsorbate_binding_indices=[0],
)

adsorbates = {
    "O": adsorbate_O,
    "OH": adsorbate_OH,
    "OOH": adsorbate_OOH,
}

adsorbate_summary = pd.DataFrame(
    [
        {
            "adsorbate": name,
            "source": "OC adsorbate database" if name in {"O", "OH"} else "explicit coordinates",
            "smiles": getattr(adsorbate, "smiles", None),
            "formula": adsorbate.atoms.get_chemical_formula(),
            "binding_indices": list(adsorbate.binding_indices),
            "n_atoms": len(adsorbate.atoms),
        }
        for name, adsorbate in adsorbates.items()
    ]
)

display(adsorbate_summary)


,adsorbate,source,smiles,formula,binding_indices,n_atoms
0,O,OC adsorbate database,*O,O,[0],1
1,OH,OC adsorbate database,*OH,HO,[0],2
2,OOH,explicit coordinates,None,HO2,[0],3


## 5. Generate AdsorbML initial structures

For each adsorbate, this cell keeps:

- every heuristic placement returned by AdsorbML;
- exactly 20 additional random surface sites generated with
  `random_site_heuristic_placement`.

No heuristic placements are truncated.


In [6]:
placement_records = []
configuration_records = []

for adsorbate_name in ADSORBATE_NAMES:
    adsorbate = adsorbates[adsorbate_name]

    heuristic_generator = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="heuristic",
    )

    random_generator = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="random_site_heuristic_placement",
        num_sites=RANDOM_SITES_PER_ADSORBATE,
    )

    groups = [
        (
            "heuristic",
            list(heuristic_generator.atoms_list),
            list(heuristic_generator.metadata_list),
        ),
        (
            "random_site",
            list(random_generator.atoms_list),
            list(random_generator.metadata_list),
        ),
    ]

    running_index = 0

    for placement_kind, atoms_list, metadata_list in groups:
        if len(atoms_list) != len(metadata_list):
            raise RuntimeError(
                f"Atoms/metadata mismatch for {adsorbate_name} {placement_kind}."
            )

        for local_index, (atoms, metadata_item) in enumerate(
            zip(atoms_list, metadata_list)
        ):
            rebuilt = rebuild_atoms(
                atoms,
                constrain_subsurface=True,
            )
            ensure_finite_geometry(
                rebuilt,
                f"{adsorbate_name} {placement_kind} {local_index}",
            )

            config_id = f"{adsorbate_name}_{running_index:03d}"

            site = np.asarray(metadata_item.get("site", [np.nan] * 3), dtype=float)
            xyz_angles = metadata_item.get("xyz_angles", None)

            configuration_records.append(
                {
                    "config_id": config_id,
                    "adsorbate": adsorbate_name,
                    "config_index": running_index,
                    "placement_kind": placement_kind,
                    "placement_local_index": local_index,
                    "initial_site_x_A": float(site[0]),
                    "initial_site_y_A": float(site[1]),
                    "initial_site_z_A": float(site[2]),
                    "sampled_angles": repr(xyz_angles),
                    "atoms": rebuilt,
                }
            )
            running_index += 1

        placement_records.append(
            {
                "adsorbate": adsorbate_name,
                "placement_kind": placement_kind,
                "n_configurations": len(atoms_list),
            }
        )

placements = pd.DataFrame(placement_records)
configuration_table = pd.DataFrame(
    [{key: value for key, value in record.items() if key != "atoms"}
     for record in configuration_records]
)

placement_summary = (
    placements
    .pivot(index="adsorbate", columns="placement_kind", values="n_configurations")
    .fillna(0)
    .astype(int)
    .reset_index()
)

placement_summary["total"] = (
    placement_summary.get("heuristic", 0)
    + placement_summary.get("random_site", 0)
)

display(placement_summary)
display(configuration_table.head())

assert (placement_summary["random_site"] == RANDOM_SITES_PER_ADSORBATE).all()


placement_kind,adsorbate,heuristic,random_site,total
0,O,7,20,27
1,OH,7,20,27
2,OOH,7,20,27


,config_id,adsorbate,config_index,placement_kind,placement_local_index,initial_site_x_A,initial_site_y_A,initial_site_z_A,sampled_angles
0,O_000,O,0,heuristic,0,1.641920e-15,2.444064e+00,18.303848,"array([0, 0, 0])"
1,O_001,O,1,heuristic,1,2.116622e+00,1.222032e+00,19.167955,"array([0, 0, 0])"
2,O_002,O,2,heuristic,2,1.058311e+00,6.110161e-01,19.600009,"array([0, 0, 0])"
3,O_003,O,3,heuristic,3,1.619421e-15,8.208959e-16,20.032062,"array([0, 0, 0])"
4,O_004,O,4,heuristic,4,2.116622e+00,3.256148e-15,18.735902,"array([0, 0, 0])"


## 6. Load the EquiformerV2 checkpoint

The model is downloaded once and cached under `/content/fairchem_checkpoints`.
The calculator predicts OC20 adsorption-referenced model scores and forces.


In [7]:
checkpoint_path = model_name_to_local_file(
    MODEL_NAME,
    local_cache=str(CHECKPOINT_DIR),
)

print("Checkpoint:", checkpoint_path)

try:
    calculator = OCPCalculator(
        checkpoint_path=checkpoint_path,
        cpu=False,
        seed=SEED,
    )
except TypeError:
    # Compatibility fallback for minor constructor differences.
    calculator = OCPCalculator(
        checkpoint_path=checkpoint_path,
        cpu=False,
    )

print("EqV2 calculator loaded.")


Checkpoint: /content/fairchem_checkpoints/eq2_31M_ec4_allmd.pt


EqV2 calculator loaded.


/usr/local/lib/python3.12/dist-packages/fairchem/core/modules/normalization/normalizer.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "mean": torch.tensor(state_dict["mean"]),


## 7. Relax the bare slab and every adsorbate–slab configuration

The clean slab is relaxed independently with EqV2 and saved as
`trajectories/bare/bare_slab_eqv2.traj`. Each adsorbate trajectory is also
independent and is written to its own `.traj` file. Failed calculations are
recorded rather than stopping the complete screening run. Existing trajectories
can be reused by leaving `OVERWRITE_EXISTING=False`.

A structure is considered converged when the final maximum atomic force is
strictly below 0.05 eV Å\(^{-1}\). The bare-slab EqV2 score is retained only as
run metadata; the adsorbate configurations are ranked directly by their EqV2
scores.


In [8]:
def maximum_force_from_array(forces: np.ndarray) -> float:
    forces = np.asarray(forces, dtype=float)
    if forces.size == 0:
        return float("nan")
    return float(np.linalg.norm(forces, axis=1).max())


def final_frame_with_results(atoms: Atoms) -> Atoms:
    """Copy a final geometry and preserve its energy and forces."""
    energy = float(atoms.get_potential_energy())
    forces = np.asarray(atoms.get_forces(), dtype=float)

    final = atoms.copy()
    final.calc = SinglePointCalculator(
        final,
        energy=energy,
        forces=forces,
    )
    return final


def read_frames(path: Path) -> list[Atoms]:
    frames = aseio.read(str(path), index=":")
    return frames if isinstance(frames, list) else [frames]


def write_complete_trajectory(
    trajectory_path: Path,
    frames: list[Atoms],
    final: Atoms,
) -> list[Atoms]:
    """Ensure traj[-1] exactly matches the returned relaxed structure."""
    if len(frames) == 0:
        frames = [final]
    elif np.max(np.abs(frames[-1].positions - final.positions)) > 1e-10:
        frames.append(final)
    else:
        frames[-1] = final

    aseio.write(
        str(trajectory_path),
        frames,
        format="traj",
    )
    return frames


def relax_bare_slab() -> dict:
    bare_traj_dir = TRAJECTORY_DIR / "bare"
    bare_log_dir = LOG_DIR / "bare"
    bare_traj_dir.mkdir(parents=True, exist_ok=True)
    bare_log_dir.mkdir(parents=True, exist_ok=True)

    trajectory_path = bare_traj_dir / "bare_slab_eqv2.traj"
    log_path = bare_log_dir / "bare_slab_eqv2.log"

    result = {
        "system_id": SYSTEM_ID,
        "structure": "bare",
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "relaxation_status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "ml_score_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
        "final_poscar": "",
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            forces = np.asarray(final.get_forces(), dtype=float)
            fmax = maximum_force_from_array(forces)

            final_poscar_dir = POSCAR_DIR / "bare_eqv2_relaxed"
            final_poscar_dir.mkdir(parents=True, exist_ok=True)
            final_poscar_path = final_poscar_dir / "POSCAR"
            aseio.write(
                final_poscar_path,
                final,
                format="vasp",
                direct=True,
                vasp5=True,
                sort=True,
            )

            return {
                **result,
                "relaxation_status": "reused",
                "n_frames": len(frames),
                "ml_score_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(
                    np.isfinite(fmax)
                    and fmax < FMAX_THRESHOLD
                ),
                "final_poscar": str(final_poscar_path),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    atoms = rebuild_atoms(
        clean_slab,
        constrain_subsurface=True,
    )
    atoms.calc = calculator

    start = time.time()

    try:
        optimizer = BFGS(
            atoms,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(
            fmax=FMAX_THRESHOLD,
            steps=MAX_RELAX_STEPS,
        )

        final = final_frame_with_results(atoms)
        frames = read_frames(trajectory_path)
        frames = write_complete_trajectory(
            trajectory_path,
            frames,
            final,
        )

        energy = float(final.get_potential_energy())
        forces = np.asarray(final.get_forces(), dtype=float)
        fmax = maximum_force_from_array(forces)

        final_poscar_dir = POSCAR_DIR / "bare_eqv2_relaxed"
        final_poscar_dir.mkdir(parents=True, exist_ok=True)
        final_poscar_path = final_poscar_dir / "POSCAR"
        aseio.write(
            final_poscar_path,
            final,
            format="vasp",
            direct=True,
            vasp5=True,
            sort=True,
        )

        return {
            **result,
            "relaxation_status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "ml_score_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(
                np.isfinite(fmax)
                and fmax < FMAX_THRESHOLD
            ),
            "final_poscar": str(final_poscar_path),
        }

    except Exception as exc:
        return {
            **result,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


bare_slab_status = relax_bare_slab()

print("Bare-slab EqV2 relaxation")
display(pd.DataFrame([bare_slab_status]))


def relax_one_configuration(record: dict) -> dict:
    adsorbate_name = record["adsorbate"]
    config_id = record["config_id"]

    adsorbate_traj_dir = TRAJECTORY_DIR / adsorbate_name
    adsorbate_log_dir = LOG_DIR / adsorbate_name
    adsorbate_traj_dir.mkdir(parents=True, exist_ok=True)
    adsorbate_log_dir.mkdir(parents=True, exist_ok=True)

    trajectory_path = adsorbate_traj_dir / f"{config_id}.traj"
    log_path = adsorbate_log_dir / f"{config_id}.log"

    base_result = {
        "system_id": SYSTEM_ID,
        "config_id": config_id,
        "adsorbate": adsorbate_name,
        "config_index": int(record["config_index"]),
        "placement_kind": record["placement_kind"],
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "relaxation_status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "ml_score_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            forces = np.asarray(final.get_forces(), dtype=float)
            fmax = maximum_force_from_array(forces)

            return {
                **base_result,
                "relaxation_status": "reused",
                "n_frames": len(frames),
                "ml_score_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(
                    np.isfinite(fmax)
                    and fmax < FMAX_THRESHOLD
                ),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    atoms = rebuild_atoms(
        record["atoms"],
        constrain_subsurface=True,
    )
    atoms.calc = calculator

    start = time.time()

    try:
        optimizer = BFGS(
            atoms,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(
            fmax=FMAX_THRESHOLD,
            steps=MAX_RELAX_STEPS,
        )

        final = final_frame_with_results(atoms)
        frames = read_frames(trajectory_path)
        frames = write_complete_trajectory(
            trajectory_path,
            frames,
            final,
        )

        energy = float(final.get_potential_energy())
        forces = np.asarray(final.get_forces(), dtype=float)
        fmax = maximum_force_from_array(forces)

        return {
            **base_result,
            "relaxation_status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "ml_score_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(
                np.isfinite(fmax)
                and fmax < FMAX_THRESHOLD
            ),
        }

    except Exception as exc:
        return {
            **base_result,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


relaxation_rows = []

for record in tqdm(
    configuration_records,
    desc="EqV2 adsorbate relaxations",
):
    relaxation_rows.append(relax_one_configuration(record))

relaxation_status = pd.DataFrame(relaxation_rows)

display(
    relaxation_status[
        [
            "adsorbate",
            "config_id",
            "placement_kind",
            "relaxation_status",
            "converged",
            "ml_score_eV",
            "final_fmax_eV_A",
            "elapsed_s",
            "error",
        ]
    ]
)

print("\nAdsorbate relaxation summary")
display(
    relaxation_status.groupby(
        ["adsorbate", "relaxation_status", "converged"],
        dropna=False,
    ).size().rename("count").reset_index()
)


Bare-slab EqV2 relaxation


,system_id,structure,trajectory,log,relaxation_status,error,elapsed_s,n_frames,ml_score_eV,final_fmax_eV_A,converged,final_poscar
0,mp-10260_111_term0,bare,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,completed,,1.888469,7,-0.754395,0.018479,True,/content/stageI_eqv2_results/mp-10260_111_term...


EqV2 adsorbate relaxations:   0%|          | 0/81 [00:00<?, ?it/s]

,adsorbate,config_id,placement_kind,relaxation_status,converged,ml_score_eV,final_fmax_eV_A,elapsed_s,error
0,O,O_000,heuristic,completed,True,2.638672,0.036443,1.654831,
1,O,O_001,heuristic,completed,True,2.304688,0.033150,3.681718,
2,O,O_002,heuristic,completed,True,1.824219,0.044507,9.623595,
3,O,O_003,heuristic,completed,True,2.181641,0.045328,2.702009,
4,O,O_004,heuristic,completed,True,1.833984,0.043955,6.935961,
...,...,...,...,...,...,...,...,...,...
76,OOH,OOH_022,random_site,completed,True,3.873047,0.042507,16.240740,
77,OOH,OOH_023,random_site,completed,True,3.888672,0.030197,10.843081,
78,OOH,OOH_024,random_site,completed,True,2.505859,0.049214,9.230363,
79,OOH,OOH_025,random_site,completed,True,3.888672,0.043816,14.834079,



Adsorbate relaxation summary


,adsorbate,relaxation_status,converged,count
0,O,completed,True,27
1,OH,completed,True,27
2,OOH,completed,True,27


## 8. Validate trajectories and export every final frame

The validator uses the OC tag convention:

- tag 0: fixed/subsurface slab atom;
- tag 1: surface slab atom;
- tag 2: adsorbate atom.

This avoids the unsafe element-based rule that would incorrectly classify
oxygen or hydrogen atoms belonging to an oxide or hydroxylated slab.

Every readable final frame, accepted or rejected, is exported to an individual
VASP `POSCAR`. A trajectory is accepted for site ranking only when it is
converged, finite, and passes the FAIRChem anomaly checks.


In [9]:
def adsorbate_indices_from_tags(atoms: Atoms, adsorbate_name: str) -> np.ndarray:
    tags = np.asarray(atoms.get_tags(), dtype=int)
    tagged = np.flatnonzero(tags == 2)
    expected = ADSORBATE_SIZES[adsorbate_name]

    if len(tagged) == expected:
        return tagged

    # Defensive fallback: AdsorbateSlabConfig appends the adsorbate atoms last.
    fallback = np.arange(len(atoms) - expected, len(atoms), dtype=int)
    tags[fallback] = 2
    atoms.set_tags(tags)
    return fallback


def anomaly_flags(frames: list[Atoms], adsorbate_name: str) -> dict:
    initial = frames[0].copy()
    final = frames[-1].copy()

    adsorbate_indices_from_tags(initial, adsorbate_name)
    adsorbate_indices_from_tags(final, adsorbate_name)

    initial_tags = np.asarray(initial.get_tags(), dtype=int)

    detector = DetectTrajAnomaly(
        initial,
        final,
        initial_tags,
    )

    return {
        "adsorbate_dissociated": bool(detector.is_adsorbate_dissociated()),
        "adsorbate_desorbed": bool(detector.is_adsorbate_desorbed()),
        "surface_changed": bool(detector.has_surface_changed()),
        "adsorbate_intercalated": bool(detector.is_adsorbate_intercalated()),
    }


def write_poscar(atoms: Atoms, directory: Path) -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    poscar_path = directory / "POSCAR"
    aseio.write(
        poscar_path,
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
        sort=True,
    )
    return poscar_path


scan_rows = []

for row in relaxation_status.itertuples(index=False):
    record = {
        "system_id": row.system_id,
        "config_id": row.config_id,
        "adsorbate": row.adsorbate,
        "config_index": int(row.config_index),
        "placement_kind": row.placement_kind,
        "trajectory": row.trajectory,
        "relaxation_status": row.relaxation_status,
        "validation_status": "rejected",
        "rejection_reason": "",
        "ml_score_eV": row.ml_score_eV,
        "final_fmax_eV_A": row.final_fmax_eV_A,
        "n_frames": int(row.n_frames),
        "final_poscar": "",
        "adsorbate_dissociated": False,
        "adsorbate_desorbed": False,
        "surface_changed": False,
        "adsorbate_intercalated": False,
    }

    if row.relaxation_status not in {"completed", "reused"}:
        record["rejection_reason"] = row.error or "relaxation_failed"
        scan_rows.append(record)
        continue

    try:
        frames = read_frames(Path(row.trajectory))
        final = frames[-1]

        all_final_dir = (
            POSCAR_DIR
            / "all_final_frames"
            / row.adsorbate
            / row.config_id
        )
        record["final_poscar"] = str(write_poscar(final, all_final_dir))

        flags = anomaly_flags(frames, row.adsorbate)
        record.update(flags)

        active_anomalies = [name for name, value in flags.items() if value]

        if not np.isfinite(row.ml_score_eV):
            record["rejection_reason"] = "non_finite_ml_score"
        elif not np.isfinite(row.final_fmax_eV_A):
            record["rejection_reason"] = "non_finite_force"
        elif row.final_fmax_eV_A >= FMAX_THRESHOLD:
            record["rejection_reason"] = (
                f"force_above_threshold:{row.final_fmax_eV_A:.6f}"
            )
        elif active_anomalies:
            record["rejection_reason"] = ";".join(active_anomalies)
        else:
            record["validation_status"] = "accepted"

    except Exception as exc:
        record["rejection_reason"] = (
            f"validation_failed:{type(exc).__name__}:{exc}"
        )

    scan_rows.append(record)

trajectory_scan = pd.DataFrame(scan_rows)

display(
    trajectory_scan[
        [
            "adsorbate",
            "config_id",
            "placement_kind",
            "validation_status",
            "rejection_reason",
            "ml_score_eV",
            "final_fmax_eV_A",
            "final_poscar",
        ]
    ]
)

print("\nValidation summary")
display(
    trajectory_scan.groupby(
        ["adsorbate", "validation_status"],
        dropna=False,
    ).size().rename("count").reset_index()
)


,adsorbate,config_id,placement_kind,validation_status,rejection_reason,ml_score_eV,final_fmax_eV_A,final_poscar
0,O,O_000,heuristic,accepted,,2.638672,0.036443,/content/stageI_eqv2_results/mp-10260_111_term...
1,O,O_001,heuristic,accepted,,2.304688,0.033150,/content/stageI_eqv2_results/mp-10260_111_term...
2,O,O_002,heuristic,accepted,,1.824219,0.044507,/content/stageI_eqv2_results/mp-10260_111_term...
3,O,O_003,heuristic,accepted,,2.181641,0.045328,/content/stageI_eqv2_results/mp-10260_111_term...
4,O,O_004,heuristic,accepted,,1.833984,0.043955,/content/stageI_eqv2_results/mp-10260_111_term...
...,...,...,...,...,...,...,...,...
76,OOH,OOH_022,random_site,accepted,,3.873047,0.042507,/content/stageI_eqv2_results/mp-10260_111_term...
77,OOH,OOH_023,random_site,accepted,,3.888672,0.030197,/content/stageI_eqv2_results/mp-10260_111_term...
78,OOH,OOH_024,random_site,rejected,adsorbate_dissociated,2.505859,0.049214,/content/stageI_eqv2_results/mp-10260_111_term...
79,OOH,OOH_025,random_site,accepted,,3.888672,0.043816,/content/stageI_eqv2_results/mp-10260_111_term...



Validation summary


,adsorbate,validation_status,count
0,O,accepted,27
1,OH,accepted,27
2,OOH,accepted,26
3,OOH,rejected,1


## 9. Classify final adsorption sites

The binding oxygen is selected from the tag-2 adsorbate atoms as the oxygen
closest to the tagged surface. The classifier then evaluates the oxygen
projection relative to nearby surface atoms and labels the final structure as
top, bridge, threefold hollow, fourfold, or a distance-based fallback.

Three levels are reported:

1. **All accepted configurations** — every valid final trajectory;
2. **Unique site instances** — configurations converging to the same explicit
   set of surface atom indices;
3. **Site families** — coordination plus sorted neighboring elements, for
   example `bridge__Ni-Sb`.

The lowest EqV2 score is retained within each group, followed by the global
minimum for each adsorbate.


In [10]:
LOCAL_XY_RADIUS = 4.8
SURFACE_LAYER_TOL = 1.8
TOP_XY_TOL = 0.45
BRIDGE_PROJECTION_TOL = 0.65
MAX_BRIDGE_LENGTH = 4.5
MAX_HOLLOW_EDGE = 4.8
BOND_BUFFER = 0.65
MAX_BOND_DISTANCE = 3.1
DISTANCE_TIE_TOL = 0.22


def mic_vectors_xy(
    atoms: Atoms,
    anchor_position: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    cell = np.asarray(atoms.cell.array, dtype=float)
    inverse_cell = np.linalg.inv(cell)

    anchor_fractional = anchor_position @ inverse_cell
    fractional = positions @ inverse_cell
    delta = fractional - anchor_fractional

    delta[:, 0] -= np.round(delta[:, 0])
    delta[:, 1] -= np.round(delta[:, 1])

    return delta @ cell


def point_segment_distance_2d(
    point: np.ndarray,
    start: np.ndarray,
    end: np.ndarray,
) -> tuple[float, float]:
    segment = end - start
    denominator = float(np.dot(segment, segment))

    if denominator < 1e-12:
        return float(np.linalg.norm(point - start)), 0.0

    parameter = float(np.dot(point - start, segment) / denominator)
    clipped = float(np.clip(parameter, 0.0, 1.0))
    closest = start + clipped * segment

    return float(np.linalg.norm(point - closest)), clipped


def point_in_triangle_2d(
    point: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    epsilon: float = 1e-8,
) -> bool:
    v0 = c - a
    v1 = b - a
    v2 = point - a

    denominator = v0[0] * v1[1] - v1[0] * v0[1]
    if abs(denominator) < 1e-12:
        return False

    u = (v2[0] * v1[1] - v1[0] * v2[1]) / denominator
    v = (v0[0] * v2[1] - v2[0] * v0[1]) / denominator

    return bool(
        u >= -epsilon
        and v >= -epsilon
        and u + v <= 1.0 + epsilon
    )


def site_type_from_coordination(coordination: int) -> str:
    return {
        1: "top",
        2: "bridge",
        3: "3-fold",
        4: "4-fold",
    }.get(
        coordination,
        "unknown" if coordination <= 0 else f"{coordination}-fold",
    )


def format_site_result(selected: list[dict], reason: str) -> dict:
    selected = sorted(selected, key=lambda item: item["index"])

    indices = [int(item["index"]) for item in selected]
    symbols = [item["symbol"] for item in selected]
    distances = [float(item["r3d"]) for item in selected]

    site_type = site_type_from_coordination(len(selected))
    neighbor_composition = "-".join(sorted(symbols))
    distance_fingerprint = "-".join(f"{value:.2f}" for value in sorted(distances))

    site_instance_key = (
        f"{site_type}__idx_"
        + "-".join(str(index) for index in indices)
    )
    site_family = f"{site_type}__{neighbor_composition}"
    local_descriptor = (
        f"{site_family}__d_{distance_fingerprint}"
    )

    return {
        "site_type": site_type,
        "coordination": len(selected),
        "neighbor_composition": neighbor_composition,
        "neighbor_indices": "-".join(str(index) for index in indices),
        "neighbor_distances_A": ";".join(f"{value:.4f}" for value in distances),
        "site_instance_key": site_instance_key,
        "site_family": site_family,
        "local_site_descriptor": local_descriptor,
        "site_assignment_reason": reason,
    }


def choose_binding_oxygen(
    atoms: Atoms,
    adsorbate_indices: np.ndarray,
    surface_indices: np.ndarray,
) -> int:
    symbols = atoms.get_chemical_symbols()
    oxygen_indices = [
        int(index)
        for index in adsorbate_indices
        if symbols[index] == "O"
    ]

    if not oxygen_indices:
        raise RuntimeError("No oxygen atom was found in the tagged adsorbate.")

    if len(oxygen_indices) == 1:
        return oxygen_indices[0]

    positions = atoms.get_positions()
    surface_positions = positions[surface_indices]

    return min(
        oxygen_indices,
        key=lambda oxygen_index: float(
            np.linalg.norm(
                mic_vectors_xy(
                    atoms,
                    positions[oxygen_index],
                    surface_positions,
                ),
                axis=1,
            ).min()
        ),
    )


def detect_adsorption_site(
    atoms: Atoms,
    adsorbate_name: str,
) -> dict:
    atoms = atoms.copy()
    adsorbate_indices = adsorbate_indices_from_tags(atoms, adsorbate_name)

    tags = np.asarray(atoms.get_tags(), dtype=int)
    surface_indices = np.flatnonzero(tags == 1)

    if len(surface_indices) == 0:
        surface_indices = np.flatnonzero(tags != 2)

    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()

    binding_oxygen = choose_binding_oxygen(
        atoms,
        adsorbate_indices,
        surface_indices,
    )
    anchor = positions[binding_oxygen]

    surface_positions = positions[surface_indices]
    vectors = mic_vectors_xy(atoms, anchor, surface_positions)
    r3d = np.linalg.norm(vectors, axis=1)
    rxy = np.linalg.norm(vectors[:, :2], axis=1)

    surface_z = surface_positions[:, 2]
    top_surface_z = float(surface_z.max())
    layer_mask = surface_z >= top_surface_z - SURFACE_LAYER_TOL
    local_mask = layer_mask & (rxy <= LOCAL_XY_RADIUS)

    local_ids = np.flatnonzero(local_mask)
    if len(local_ids) == 0:
        local_ids = np.argsort(r3d)[:12]

    local_ids = sorted(
        local_ids,
        key=lambda index: (rxy[index], r3d[index]),
    )[:16]

    candidates = [
        {
            "index": int(surface_indices[local_index]),
            "symbol": symbols[int(surface_indices[local_index])],
            "xy": vectors[local_index, :2],
            "rxy": float(rxy[local_index]),
            "r3d": float(r3d[local_index]),
        }
        for local_index in local_ids
    ]

    origin = np.zeros(2)
    nearest = min(candidates, key=lambda item: item["rxy"])

    if nearest["rxy"] <= TOP_XY_TOL:
        result = format_site_result([nearest], "projection_top")
        result["binding_oxygen_index"] = binding_oxygen
        return result

    bridge_options = []

    for atom_a, atom_b in itertools.combinations(candidates, 2):
        pair_length = float(np.linalg.norm(atom_a["xy"] - atom_b["xy"]))
        if pair_length > MAX_BRIDGE_LENGTH:
            continue

        distance, parameter = point_segment_distance_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
        )

        if distance <= BRIDGE_PROJECTION_TOL and 0.0 <= parameter <= 1.0:
            score = (
                distance
                + 0.20 * abs(parameter - 0.5)
                + 0.03 * (atom_a["r3d"] + atom_b["r3d"])
            )
            bridge_options.append((score, atom_a, atom_b))

    if bridge_options:
        bridge_options.sort(key=lambda item: item[0])
        _, atom_a, atom_b = bridge_options[0]
        result = format_site_result(
            [atom_a, atom_b],
            "projection_bridge",
        )
        result["binding_oxygen_index"] = binding_oxygen
        return result

    hollow_options = []

    for atom_a, atom_b, atom_c in itertools.combinations(candidates, 3):
        edge_lengths = [
            float(np.linalg.norm(atom_a["xy"] - atom_b["xy"])),
            float(np.linalg.norm(atom_a["xy"] - atom_c["xy"])),
            float(np.linalg.norm(atom_b["xy"] - atom_c["xy"])),
        ]

        if max(edge_lengths) > MAX_HOLLOW_EDGE:
            continue

        if point_in_triangle_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
            atom_c["xy"],
        ):
            centroid = (atom_a["xy"] + atom_b["xy"] + atom_c["xy"]) / 3.0
            score = (
                float(np.linalg.norm(centroid))
                + 0.02 * (
                    atom_a["r3d"]
                    + atom_b["r3d"]
                    + atom_c["r3d"]
                )
            )
            hollow_options.append((score, atom_a, atom_b, atom_c))

    if hollow_options:
        hollow_options.sort(key=lambda item: item[0])
        _, atom_a, atom_b, atom_c = hollow_options[0]
        result = format_site_result(
            [atom_a, atom_b, atom_c],
            "projection_hollow",
        )
        result["binding_oxygen_index"] = binding_oxygen
        return result

    oxygen_radius = covalent_radii[atomic_numbers["O"]]
    bonded = []

    for candidate in candidates:
        surface_radius = covalent_radii[
            atomic_numbers[candidate["symbol"]]
        ]
        expected = oxygen_radius + surface_radius
        normalized = candidate["r3d"] / expected if expected > 0 else candidate["r3d"]

        if (
            candidate["r3d"] <= expected + BOND_BUFFER
            or candidate["r3d"] <= MAX_BOND_DISTANCE
        ):
            bonded.append((normalized, candidate))

    if bonded:
        minimum_score = min(item[0] for item in bonded)
        selected = [
            item[1]
            for item in bonded
            if item[0] <= minimum_score + DISTANCE_TIE_TOL
        ]
        selected = sorted(selected, key=lambda item: item["r3d"])[:4]

        result = format_site_result(
            selected,
            "bond_distance_fallback",
        )
        result["binding_oxygen_index"] = binding_oxygen
        return result

    result = format_site_result(
        [nearest],
        "nearest_projection_fallback",
    )
    result["binding_oxygen_index"] = binding_oxygen
    return result


In [11]:
accepted_rows = trajectory_scan[
    trajectory_scan["validation_status"] == "accepted"
].copy()

site_rows = []

for row in accepted_rows.itertuples(index=False):
    final = aseio.read(row.trajectory, index=-1)
    site = detect_adsorption_site(final, row.adsorbate)

    site_rows.append(
        {
            "system_id": row.system_id,
            "config_id": row.config_id,
            "adsorbate": row.adsorbate,
            "config_index": int(row.config_index),
            "placement_kind": row.placement_kind,
            "trajectory": row.trajectory,
            "final_poscar": row.final_poscar,
            "ml_score_eV": float(row.ml_score_eV),
            "final_fmax_eV_A": float(row.final_fmax_eV_A),
            **site,
        }
    )

all_adsorption_sites = pd.DataFrame(site_rows)

if all_adsorption_sites.empty:
    unique_site_minima = pd.DataFrame()
    site_family_minima = pd.DataFrame()
    global_minima = pd.DataFrame()
    print("No accepted trajectory is available for site analysis.")
else:
    all_adsorption_sites["delta_from_adsorbate_min_eV"] = (
        all_adsorption_sites["ml_score_eV"]
        - all_adsorption_sites.groupby("adsorbate")["ml_score_eV"].transform("min")
    )

    unique_site_minima = (
        all_adsorption_sites
        .sort_values("ml_score_eV")
        .groupby(
            ["system_id", "adsorbate", "site_instance_key"],
            as_index=False,
        )
        .first()
        .sort_values(["adsorbate", "ml_score_eV"])
        .reset_index(drop=True)
    )

    site_family_minima = (
        all_adsorption_sites
        .sort_values("ml_score_eV")
        .groupby(
            ["system_id", "adsorbate", "site_family"],
            as_index=False,
        )
        .first()
        .sort_values(["adsorbate", "ml_score_eV"])
        .reset_index(drop=True)
    )

    global_minima = (
        all_adsorption_sites
        .sort_values("ml_score_eV")
        .groupby(["system_id", "adsorbate"], as_index=False)
        .first()
        .sort_values("adsorbate")
        .reset_index(drop=True)
    )

print("All accepted final adsorption sites")
display(all_adsorption_sites)

print("Minimum per unique site instance")
display(unique_site_minima)

print("Minimum per site family")
display(site_family_minima)

print("Global minimum per adsorbate")
display(global_minima)


All accepted final adsorption sites


,system_id,config_id,adsorbate,config_index,placement_kind,trajectory,final_poscar,ml_score_eV,final_fmax_eV_A,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0,O_000,O,0,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,2.638672,0.036443,top,1,Ni,2,1.9424,top__idx_2,top__Ni,top__Ni__d_1.94,projection_top,48,0.876953
1,mp-10260_111_term0,O_001,O,1,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,2.304688,0.033150,top,1,Ni,4,1.8746,top__idx_4,top__Ni,top__Ni__d_1.87,projection_top,48,0.542969
2,mp-10260_111_term0,O_002,O,2,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.824219,0.044507,bridge,2,Ni-Sb,4-9,1.9020;1.9879,bridge__idx_4-9,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.062500
3,mp-10260_111_term0,O_003,O,3,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,2.181641,0.045328,top,1,Sb,9,1.8693,top__idx_9,top__Sb,top__Sb__d_1.87,projection_top,48,0.419922
4,mp-10260_111_term0,O_004,O,4,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.833984,0.043955,bridge,2,Ni-Sb,4-9,1.8934;1.9901,bridge__idx_4-9,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.072266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,mp-10260_111_term0,OOH_021,OOH,21,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.876953,0.049056,top,1,Sb,33,2.1196,top__idx_33,top__Sb,top__Sb__d_2.12,projection_top,48,0.003906
76,mp-10260_111_term0,OOH_022,OOH,22,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.873047,0.042507,top,1,Sb,45,2.1179,top__idx_45,top__Sb,top__Sb__d_2.12,projection_top,48,0.000000
77,mp-10260_111_term0,OOH_023,OOH,23,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.888672,0.030197,top,1,Sb,33,2.1227,top__idx_33,top__Sb,top__Sb__d_2.12,projection_top,48,0.015625
78,mp-10260_111_term0,OOH_025,OOH,25,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.888672,0.043816,top,1,Sb,9,2.1213,top__idx_9,top__Sb,top__Sb__d_2.12,projection_top,48,0.015625


Minimum per unique site instance


,system_id,adsorbate,site_instance_key,config_id,config_index,placement_kind,trajectory,final_poscar,ml_score_eV,final_fmax_eV_A,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0,O,bridge__idx_2-28,O_016,16,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.761719,0.039647,bridge,2,Ni-Ni,2-28,1.9549;1.8802,bridge__Ni-Ni,bridge__Ni-Ni__d_1.88-1.95,projection_bridge,48,0.000000
1,mp-10260_111_term0,O,bridge__idx_4-9,O_002,2,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.824219,0.044507,bridge,2,Ni-Sb,4-9,1.9020;1.9879,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.062500
2,mp-10260_111_term0,O,bridge__idx_9-28,O_018,18,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.826172,0.043596,bridge,2,Ni-Sb,9-28,1.9876;1.8992,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.064453
3,mp-10260_111_term0,O,bridge__idx_9-40,O_017,17,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.826172,0.041759,bridge,2,Ni-Sb,9-40,1.9891;1.8981,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.064453
4,mp-10260_111_term0,O,bridge__idx_16-21,O_013,13,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.832031,0.036544,bridge,2,Ni-Sb,16-21,1.9069;1.9870,bridge__Ni-Sb,bridge__Ni-Sb__d_1.91-1.99,projection_bridge,48,0.070312
5,mp-10260_111_term0,O,bridge__idx_40-45,O_026,26,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.833984,0.047309,bridge,2,Ni-Sb,40-45,1.8941;1.9918,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.072266
6,mp-10260_111_term0,O,bridge__idx_16-33,O_012,12,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.835938,0.046835,bridge,2,Ni-Sb,16-33,1.8883;1.9912,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.074219
7,mp-10260_111_term0,O,bridge__idx_16-45,O_020,20,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.835938,0.040644,bridge,2,Ni-Sb,16-45,1.8981;1.9927,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.074219
8,mp-10260_111_term0,O,bridge__idx_4-33,O_021,21,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.835938,0.029373,bridge,2,Ni-Sb,4-33,1.8859;1.9897,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.074219
9,mp-10260_111_term0,O,bridge__idx_4-45,O_023,23,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.839844,0.046625,bridge,2,Ni-Sb,4-45,1.8919;1.9904,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.078125


Minimum per site family


,system_id,adsorbate,site_family,config_id,config_index,placement_kind,trajectory,final_poscar,ml_score_eV,final_fmax_eV_A,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0,O,bridge__Ni-Ni,O_016,16,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.761719,0.039647,bridge,2,Ni-Ni,2-28,1.9549;1.8802,bridge__idx_2-28,bridge__Ni-Ni__d_1.88-1.95,projection_bridge,48,0.000000
1,mp-10260_111_term0,O,bridge__Ni-Sb,O_002,2,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.824219,0.044507,bridge,2,Ni-Sb,4-9,1.9020;1.9879,bridge__idx_4-9,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.062500
2,mp-10260_111_term0,O,top__Sb,O_025,25,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,2.179688,0.042958,top,1,Sb,9,1.8709,top__idx_9,top__Sb__d_1.87,projection_top,48,0.417969
3,mp-10260_111_term0,O,top__Ni,O_001,1,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,2.304688,0.033150,top,1,Ni,4,1.8746,top__idx_4,top__Ni__d_1.87,projection_top,48,0.542969
4,mp-10260_111_term0,OH,top__Sb,OH_021,21,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,0.803223,0.035472,top,1,Sb,9,2.0397,top__idx_9,top__Sb__d_2.04,projection_top,48,0.000000
5,mp-10260_111_term0,OH,bridge__Ni-Sb,OH_007,7,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.093750,0.042500,bridge,2,Ni-Sb,21-28,2.2035;2.1553,bridge__idx_21-28,bridge__Ni-Sb__d_2.16-2.20,projection_bridge,48,0.290527
6,mp-10260_111_term0,OOH,top__Sb,OOH_022,22,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.873047,0.042507,top,1,Sb,45,2.1179,top__idx_45,top__Sb__d_2.12,projection_top,48,0.000000
7,mp-10260_111_term0,OOH,bridge__Sb-Sb,OOH_018,18,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.888672,0.042698,bridge,2,Sb-Sb,21-33,4.2343;2.1168,bridge__idx_21-33,bridge__Sb-Sb__d_2.12-4.23,projection_bridge,48,0.015625
8,mp-10260_111_term0,OOH,bridge__Ni-Sb,OOH_001,1,heuristic,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.900391,0.045958,bridge,2,Ni-Sb,9-40,2.1294;3.5370,bridge__idx_9-40,bridge__Ni-Sb__d_2.13-3.54,projection_bridge,48,0.027344
9,mp-10260_111_term0,OOH,bridge__Ni-Ni,OOH_013,13,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,4.156250,0.049189,bridge,2,Ni-Ni,16-28,4.0429;2.4733,bridge__idx_16-28,bridge__Ni-Ni__d_2.47-4.04,projection_bridge,48,0.283203


Global minimum per adsorbate


,system_id,adsorbate,config_id,config_index,placement_kind,trajectory,final_poscar,ml_score_eV,final_fmax_eV_A,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0,O,O_016,16,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,1.761719,0.039647,bridge,2,Ni-Ni,2-28,1.9549;1.8802,bridge__idx_2-28,bridge__Ni-Ni,bridge__Ni-Ni__d_1.88-1.95,projection_bridge,48,0.0
1,mp-10260_111_term0,OH,OH_021,21,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,0.803223,0.035472,top,1,Sb,9,2.0397,top__idx_9,top__Sb,top__Sb__d_2.04,projection_top,48,0.0
2,mp-10260_111_term0,OOH,OOH_022,22,random_site,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,3.873047,0.042507,top,1,Sb,45,2.1179,top__idx_45,top__Sb,top__Sb__d_2.12,projection_top,48,0.0


## 10. Interactive py3Dmol visualization

The selector below uses one py3Dmol output area. It contains the initial clean
slab, the independently EqV2-relaxed bare slab, every global adsorbate minimum,
and every unique-site minimum. Selecting a new entry replaces the previous
molecular viewer rather than creating many static plots.


In [12]:
# Interactive py3Dmol structure viewer.

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass


def atoms_to_cif_string(atoms: Atoms) -> str:
    """Convert an ASE Atoms object to a CIF-formatted string."""
    buffer = io.BytesIO()
    aseio.write(buffer, atoms, format="cif")
    return buffer.getvalue().decode("utf-8")


def show_atoms_py3dmol(
    atoms: Atoms,
    *,
    width: int = 850,
    height: int = 520,
) -> None:
    """Display an ASE Atoms object using py3Dmol."""
    viewer = py3Dmol.view(
        width=width,
        height=height,
    )
    viewer.addModel(
        atoms_to_cif_string(atoms),
        "cif",
    )
    viewer.setStyle(
        {
            "sphere": {"scale": 0.32},
            "stick": {"radius": 0.14},
        }
    )
    viewer.addUnitCell()
    viewer.zoomTo()
    viewer.show()


structure_catalog = {
    "Clean slab | initial": str(clean_traj_path),
}

bare_trajectory_path = Path(
    bare_slab_status.get("trajectory", "")
)
if bare_trajectory_path.exists():
    structure_catalog[
        "Bare slab | EqV2-relaxed"
    ] = str(bare_trajectory_path)


if not global_minima.empty:
    for row in global_minima.itertuples(index=False):
        label = (
            f"GLOBAL MIN | {row.adsorbate}* | "
            f"{row.site_family} | "
            f"{row.ml_score_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)


if not unique_site_minima.empty:
    for row in unique_site_minima.itertuples(index=False):
        label = (
            f"UNIQUE SITE | {row.adsorbate}* | "
            f"{row.site_instance_key} | "
            f"{row.ml_score_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)


default_structure = (
    "Bare slab | EqV2-relaxed"
    if "Bare slab | EqV2-relaxed" in structure_catalog
    else "Clean slab | initial"
)

selector = widgets.Dropdown(
    options=list(structure_catalog.keys()),
    value=default_structure,
    description="Structure:",
    layout=widgets.Layout(width="95%"),
)

viewer_output = widgets.Output()


def render_selected_structure(change=None) -> None:
    """Load and display the structure selected in the dropdown."""
    label = selector.value
    trajectory_path = Path(structure_catalog[label])

    with viewer_output:
        clear_output(wait=True)

        if not trajectory_path.exists():
            print(f"Structure file not found: {trajectory_path}")
            return

        try:
            atoms = aseio.read(
                str(trajectory_path),
                index=-1,
            )

            print(label)
            show_atoms_py3dmol(atoms)

        except Exception as exc:
            print(f"Could not display {label}")
            print(f"{type(exc).__name__}: {exc}")


selector.observe(
    render_selected_structure,
    names="value",
)

display(
    widgets.VBox(
        [
            selector,
            viewer_output,
        ]
    )
)

render_selected_structure()


## 11. Export tables, selected structures, and article-repository VASP inputs

Every readable adsorbate trajectory already has a `POSCAR` under
`poscars/all_final_frames/`. This final export additionally creates:

- the lowest-scoring structure for each explicit unique site;
- the global EqV2 minimum for O*, OH*, and OOH*;
- `vasp/bare`, `vasp/O`, `vasp/OH`, and `vasp/OOH`;
- `POSCAR`, `INCAR`, `KPOINTS`, `job.sh`, `metadata.json`, `POTCAR.spec`,
  and a non-distributable `POTCAR` placeholder in each VASP folder;
- `vasp/manifest.csv` and `vasp/README.md`;
- CSV summaries for placements, relaxations, validation, and site analysis;
- one ZIP archive containing the complete result directory.

The supplied INCAR represents a VASP **single-point validation** calculation on
the final EqV2 geometry (`NSW = 0`). Replace the placeholder POTCAR privately
before running VASP.


In [13]:
def export_selected_poscars(
    table: pd.DataFrame,
    category: str,
) -> pd.DataFrame:
    exported = []

    if table.empty:
        return pd.DataFrame()

    for row in table.itertuples(index=False):
        final = aseio.read(
            row.trajectory,
            index=-1,
        )

        if category == "unique_site_minima":
            folder_name = (
                f"{row.config_id}__"
                f"{row.site_instance_key.replace('/', '_')}"
            )
        else:
            folder_name = (
                f"{row.adsorbate}_global_min__"
                f"{row.config_id}"
            )

        target_directory = (
            POSCAR_DIR
            / category
            / row.adsorbate
            / folder_name
        )
        poscar_path = write_poscar(
            final,
            target_directory,
        )

        exported.append(
            {
                "category": category,
                "adsorbate": row.adsorbate,
                "config_id": row.config_id,
                "site_type": row.site_type,
                "site_family": row.site_family,
                "site_instance_key": row.site_instance_key,
                "ml_score_eV": row.ml_score_eV,
                "trajectory": row.trajectory,
                "poscar": str(poscar_path),
            }
        )

    return pd.DataFrame(exported)


unique_poscars = export_selected_poscars(
    unique_site_minima,
    "unique_site_minima",
)
global_poscars = export_selected_poscars(
    global_minima,
    "global_minima",
)
selected_poscars = pd.concat(
    [unique_poscars, global_poscars],
    ignore_index=True,
)


def write_poscar_with_selective_dynamics(
    poscar_path: Path,
    atoms: Atoms,
) -> list[str]:
    """Write a deterministic VASP5 POSCAR and preserve FixAtoms flags."""
    symbols = atoms.get_chemical_symbols()

    element_order = []
    for symbol in symbols:
        if symbol not in element_order:
            element_order.append(symbol)

    grouped_indices = [
        index
        for element in element_order
        for index, symbol in enumerate(symbols)
        if symbol == element
    ]
    counts = [
        sum(symbol == element for symbol in symbols)
        for element in element_order
    ]

    fixed_indices = set()
    for constraint in getattr(
        atoms,
        "constraints",
        [],
    ):
        if isinstance(constraint, FixAtoms):
            fixed_indices.update(
                map(int, constraint.get_indices())
            )

    scaled_positions = atoms.get_scaled_positions(
        wrap=False,
    )
    cell = np.asarray(
        atoms.cell.array,
        dtype=float,
    )

    lines = [
        f"{SYSTEM_ID} final EqV2 frame",
        "1.0",
    ]
    lines.extend(
        "  "
        + " ".join(
            f"{component:.12f}"
            for component in vector
        )
        for vector in cell
    )
    lines.append(
        "  " + " ".join(element_order)
    )
    lines.append(
        "  " + " ".join(map(str, counts))
    )
    lines.append("Selective dynamics")
    lines.append("Direct")

    for index in grouped_indices:
        position = scaled_positions[index]
        flags = (
            "F F F"
            if index in fixed_indices
            else "T T T"
        )
        lines.append(
            "  "
            + " ".join(
                f"{component:.12f}"
                for component in position
            )
            + f"  {flags}"
        )

    poscar_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )
    return element_order


def write_kpoints(
    directory: Path,
    atoms: Atoms,
) -> tuple[int, int, int]:
    cell = np.asarray(
        atoms.cell.array,
        dtype=float,
    )
    a_length = float(np.linalg.norm(cell[0]))
    b_length = float(np.linalg.norm(cell[1]))

    if a_length <= 0.0 or b_length <= 0.0:
        raise ValueError(
            "Invalid in-plane lattice vectors for KPOINTS."
        )

    kx = max(
        1,
        int(round(KPOINT_DENSITY / a_length)),
    )
    ky = max(
        1,
        int(round(KPOINT_DENSITY / b_length)),
    )
    kz = 1

    text = (
        "Automatic mesh\n"
        "0\n"
        "Monkhorst-Pack\n"
        f"{kx} {ky} {kz}\n"
        "0 0 0\n"
    )
    (directory / "KPOINTS").write_text(
        text,
        encoding="utf-8",
    )
    return kx, ky, kz


def write_incar(
    directory: Path,
    system_label: str,
) -> None:
    text = f"""SYSTEM = {system_label}
PREC = Accurate
ENCUT = 350
GGA = RP
EDIFF = 1E-5
NELM = 120

# Single-point evaluation of the final EqV2 geometry
IBRION = -1
NSW = 0
ISIF = 2

ISYM = 0
SYMPREC = 1E-10
LREAL = Auto
LASPH = .TRUE.
LWAVE = .FALSE.
LCHARG = .FALSE.
NCORE = 4
"""
    (directory / "INCAR").write_text(
        text,
        encoding="utf-8",
    )


def safe_job_name(label: str) -> str:
    cleaned = re.sub(
        r"[^A-Za-z0-9_-]+",
        "-",
        label,
    ).strip("-")
    return cleaned[:64] or "vasp_SP"


def write_batch_script(
    directory: Path,
    job_label: str,
) -> None:
    job_name = safe_job_name(job_label)
    text = f"""#!/bin/bash
#SBATCH -N 1
#SBATCH -n 64
#SBATCH --time=00:59:00
#SBATCH -p rome
#SBATCH -J {job_name}
#SBATCH --output=out.%j
#SBATCH --error=err.%j

set -euo pipefail

echo "SLURM_JOBID ${{SLURM_JOBID:-unset}}"
echo "SLURM_NODELIST ${{SLURM_NODELIST:-unset}}"

if grep -q "POTCAR NOT DISTRIBUTED" POTCAR; then
    echo "ERROR: Replace the repository POTCAR placeholder with a licensed POTCAR."
    exit 2
fi

module purge
module load 2024
module load VASP6/6.4.3-foss-2024a

srun vasp_std > vasp.out
"""
    script_path = directory / "job.sh"
    script_path.write_text(
        text,
        encoding="utf-8",
    )
    script_path.chmod(0o755)


def write_potcar_placeholder(
    directory: Path,
    elements: list[str],
) -> None:
    element_line = " ".join(elements)

    placeholder = f"""POTCAR NOT DISTRIBUTED

This is an intentional repository placeholder, not a VASP potential file.
Required POSCAR/POTCAR element order:
{element_line}

Create the real POTCAR privately from your licensed VASP PAW-PBE library.
Do not commit or redistribute the licensed POTCAR.
"""
    (directory / "POTCAR").write_text(
        placeholder,
        encoding="utf-8",
    )

    spec = (
        "# Required POTCAR order; one element per line.\n"
        + "\n".join(elements)
        + "\n"
    )
    (directory / "POTCAR.spec").write_text(
        spec,
        encoding="utf-8",
    )


def export_vasp_case(
    case_name: str,
    trajectory_path: str,
    metadata: dict,
) -> dict:
    source_path = Path(trajectory_path)

    if not source_path.exists():
        raise FileNotFoundError(
            f"Trajectory not found for {case_name}: "
            f"{source_path}"
        )

    atoms = aseio.read(
        str(source_path),
        index=-1,
    )

    run_directory = VASP_DIR / case_name
    if run_directory.exists():
        shutil.rmtree(run_directory)
    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    poscar_path = run_directory / "POSCAR"
    elements = write_poscar_with_selective_dynamics(
        poscar_path,
        atoms,
    )
    kmesh = write_kpoints(
        run_directory,
        atoms,
    )
    write_incar(
        run_directory,
        f"{SYSTEM_ID}_{case_name}_EqV2_final",
    )
    write_batch_script(
        run_directory,
        f"{SYSTEM_ID}_{case_name}_SP",
    )
    write_potcar_placeholder(
        run_directory,
        elements,
    )

    case_metadata = {
        "system_id": SYSTEM_ID,
        "case": case_name,
        "source_trajectory": str(source_path),
        "source_frame": -1,
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": elements,
        "kpoint_mesh": list(kmesh),
        "kpoint_density_parameter": KPOINT_DENSITY,
        "vasp_run_mode": VASP_RUN_MODE,
        "potcar_distributed": False,
        **metadata,
    }

    with (run_directory / "metadata.json").open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            case_metadata,
            handle,
            indent=2,
        )

    return {
        "case": case_name,
        "directory": str(run_directory),
        "poscar": str(poscar_path),
        "trajectory": str(source_path),
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": " ".join(elements),
        "kpoints": " ".join(map(str, kmesh)),
        "ml_score_eV": metadata.get(
            "ml_score_eV",
            np.nan,
        ),
        "config_id": metadata.get(
            "config_id",
            "",
        ),
        "site_family": metadata.get(
            "site_family",
            "",
        ),
    }


vasp_records = []

bare_trajectory = Path(
    bare_slab_status.get("trajectory", "")
)
if (
    bare_slab_status.get("relaxation_status")
    in {"completed", "reused"}
    and bare_trajectory.exists()
):
    vasp_records.append(
        export_vasp_case(
            "bare",
            str(bare_trajectory),
            {
                "structure_type": "bare_slab",
                "relaxation_status": bare_slab_status[
                    "relaxation_status"
                ],
                "converged": bool(
                    bare_slab_status["converged"]
                ),
                "ml_score_eV": float(
                    bare_slab_status["ml_score_eV"]
                ),
                "final_fmax_eV_A": float(
                    bare_slab_status["final_fmax_eV_A"]
                ),
            },
        )
    )
else:
    print(
        "[WARN] Bare slab was not exported to VASP "
        "because no readable completed trajectory exists."
    )


available_global_adsorbates = set(
    global_minima.get(
        "adsorbate",
        pd.Series(dtype=str),
    ).tolist()
)
missing_global_adsorbates = (
    set(ADSORBATE_NAMES)
    - available_global_adsorbates
)
if missing_global_adsorbates:
    print(
        "[WARN] No accepted global minimum for:",
        ", ".join(sorted(missing_global_adsorbates)),
    )


for adsorbate_name in ADSORBATE_NAMES:
    if (
        global_minima.empty
        or "adsorbate" not in global_minima.columns
    ):
        selected = pd.DataFrame()
    else:
        selected = global_minima[
            global_minima["adsorbate"]
            == adsorbate_name
        ]

    if selected.empty:
        continue

    row = selected.iloc[0]
    vasp_records.append(
        export_vasp_case(
            adsorbate_name,
            str(row["trajectory"]),
            {
                "structure_type": "adsorbate_global_minimum",
                "adsorbate": adsorbate_name,
                "config_id": str(row["config_id"]),
                "placement_kind": str(
                    row["placement_kind"]
                ),
                "site_type": str(row["site_type"]),
                "site_family": str(row["site_family"]),
                "site_instance_key": str(
                    row["site_instance_key"]
                ),
                "neighbor_composition": str(
                    row["neighbor_composition"]
                ),
                "ml_score_eV": float(
                    row["ml_score_eV"]
                ),
                "final_fmax_eV_A": float(
                    row["final_fmax_eV_A"]
                ),
            },
        )
    )


vasp_manifest = pd.DataFrame(
    vasp_records,
)

readme_text = f"""# VASP inputs from final EqV2 structures

System: `{SYSTEM_ID}`

This directory contains one folder for the independently EqV2-relaxed bare
slab and one folder for each accepted adsorbate global minimum:

- `bare/`
- `O/`
- `OH/`
- `OOH/`

Each folder contains:

- `POSCAR`: final frame (`traj[-1]`) from the selected EqV2 trajectory;
- `INCAR`: RPBE single-point settings (`NSW = 0`);
- `KPOINTS`: Monkhorst-Pack mesh generated with
  `round({KPOINT_DENSITY}/|a|) × round({KPOINT_DENSITY}/|b|) × 1`;
- `job.sh`: example DIFFER/Snellius-style Slurm script;
- `metadata.json`: source trajectory, model score, force, and site metadata;
- `POTCAR.spec`: required element order;
- `POTCAR`: deliberately invalid licensing placeholder.

## Important POTCAR notice

VASP PAW datasets are licensed and are not included in this repository.
Before running VASP, replace each placeholder `POTCAR` privately with the
licensed PAW-PBE datasets in exactly the order listed in `POTCAR.spec`.
Never commit or redistribute the licensed replacement file.

The included `job.sh` checks for the placeholder text and exits before launching
VASP when the real POTCAR has not been supplied.
"""
(VASP_DIR / "README.md").write_text(
    readme_text,
    encoding="utf-8",
)
vasp_manifest.to_csv(
    VASP_DIR / "manifest.csv",
    index=False,
)


tables = {
    "adsorbate_summary.csv": adsorbate_summary,
    "placement_summary.csv": placement_summary,
    "initial_configuration_metadata.csv": configuration_table,
    "bare_slab_relaxation.csv": pd.DataFrame(
        [bare_slab_status]
    ),
    "relaxation_status.csv": relaxation_status,
    "trajectory_validation.csv": trajectory_scan,
    "all_adsorption_sites.csv": all_adsorption_sites,
    "unique_site_minima.csv": unique_site_minima,
    "site_family_minima.csv": site_family_minima,
    "global_minima.csv": global_minima,
    "selected_poscars.csv": selected_poscars,
    "vasp_manifest.csv": vasp_manifest,
}

for filename, table in tables.items():
    table.to_csv(
        TABLE_DIR / filename,
        index=False,
    )


run_metadata = {
    "bulk_id": BULK_ID,
    "miller_index": MILLER_INDEX,
    "surface_index": SURFACE_INDEX,
    "bulk_formula": bulk.atoms.get_chemical_formula(),
    "slab_formula": clean_slab.get_chemical_formula(),
    "model_name": MODEL_NAME,
    "random_sites_per_adsorbate": RANDOM_SITES_PER_ADSORBATE,
    "fmax_threshold_eV_A": FMAX_THRESHOLD,
    "max_relax_steps": MAX_RELAX_STEPS,
    "bare_slab_relaxed": bool(
        bare_slab_status.get(
            "relaxation_status"
        ) in {"completed", "reused"}
    ),
    "bare_slab_converged": bool(
        bare_slab_status.get(
            "converged",
            False,
        )
    ),
    "bare_slab_ml_score_eV": (
        float(bare_slab_status["ml_score_eV"])
        if np.isfinite(
            bare_slab_status.get(
                "ml_score_eV",
                np.nan,
            )
        )
        else None
    ),
    "bare_slab_energy_used_for_adsorption_energy": False,
    "vasp_run_mode": VASP_RUN_MODE,
    "vasp_cases_exported": vasp_manifest[
        "case"
    ].tolist() if not vasp_manifest.empty else [],
    "potcar_distributed": False,
    "o_source": "OC adsorbate database: *O",
    "oh_source": "OC adsorbate database: *OH",
    "ooh_source": (
        "explicit O-O-H coordinates; "
        "binding atom index 0"
    ),
}

with (SYSTEM_DIR / "run_metadata.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        run_metadata,
        handle,
        indent=2,
    )


archive_path = shutil.make_archive(
    base_name=str(
        ROOT_DIR
        / f"{SYSTEM_ID}_stageI_eqv2_repository"
    ),
    format="zip",
    root_dir=str(ROOT_DIR),
    base_dir=SYSTEM_ID,
)

print("Result directory:", SYSTEM_DIR)
print("ZIP archive:", archive_path)
print("\nVASP cases:")
display(vasp_manifest)

print("\nTop-level outputs:")
for path in sorted(SYSTEM_DIR.iterdir()):
    print(" -", path.name)


Result directory: /content/stageI_eqv2_results/mp-10260_111_term0
ZIP archive: /content/stageI_eqv2_results/mp-10260_111_term0_stageI_eqv2_repository.zip

VASP cases:


,case,directory,poscar,trajectory,formula,n_atoms,element_order,kpoints,ml_score_eV,config_id,site_family
0,bare,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,Ni36Sb12,48,Ni Sb,5 5 1,-0.754395,,
1,O,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,Ni36OSb12,49,Ni Sb O,5 5 1,1.761719,O_016,bridge__Ni-Ni
2,OH,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,HNi36OSb12,50,Ni Sb O H,5 5 1,0.803223,OH_021,top__Sb
3,OOH,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,HNi36O2Sb12,51,Ni Sb O H,5 5 1,3.873047,OOH_022,top__Sb



Top-level outputs:
 - clean_slab_unrelaxed.traj
 - logs
 - poscars
 - run_metadata.json
 - tables
 - trajectories
 - vasp


In [15]:
# Optional Colab download.

from pathlib import Path

archive_path = Path(archive_path)

try:
    from google.colab import files

    files.download(str(archive_path))
except ImportError:
    print("Archive available at:", archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Output interpretation

- `tables/bare_slab_relaxation.csv` records the independent EqV2 relaxation of
  the constrained clean slab.
- `tables/trajectory_validation.csv` contains every attempted adsorbate
  configuration and the reason for rejection, where applicable.
- `tables/all_adsorption_sites.csv` contains every accepted final adsorbate
  configuration.
- `tables/unique_site_minima.csv` removes trajectories that converged to the
  same explicit surface-atom site and retains the lowest EqV2 score.
- `tables/site_family_minima.csv` groups sites by coordination and neighboring
  elements.
- `tables/global_minima.csv` contains one lowest-scoring structure for O*, OH*,
  and OOH*.
- `poscars/all_final_frames/` contains `traj[-1] → POSCAR` exports for every
  readable adsorbate trajectory.
- `poscars/bare_eqv2_relaxed/` contains the final EqV2-relaxed bare slab.
- `poscars/unique_site_minima/` and `poscars/global_minima/` contain the
  structures selected during screening.
- `vasp/bare/`, `vasp/O/`, `vasp/OH/`, and `vasp/OOH/` contain the final
  repository-ready VASP input packages.
- `vasp/manifest.csv` records the source trajectory and selection metadata for
  each exported VASP folder.

The VASP inputs are intentionally non-runnable until the placeholder `POTCAR`
in each folder is replaced privately with licensed PAW-PBE datasets in the
order specified by `POTCAR.spec`.


## References

1. Lan, J.; Palizhati, A.; Shuaibi, M.; *et al.*  
   **AdsorbML: A Leap in Efficiency for Adsorption Energy Calculations Using
   Generalizable Machine Learning Potentials.**  
   *npj Computational Materials* **2023**.

2. Liao, Y.-L.; Wood, B. M.; Das, A.; Smidt, T.  
   **EquiformerV2: Improved Equivariant Transformer for Scaling to
   Higher-Degree Representations.**  
   *ICLR* **2024**.

3. Chanussot, L.; Das, A.; Goyal, S.; *et al.*  
   **Open Catalyst 2020 (OC20) Dataset and Community Challenges.**  
   *ACS Catalysis* **2021**, *11*, 6059–6072.
